In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
MC-dropout analogue of analyze_seed_stability_10_kendall.ipynb: same
plateau-diagnostic plot (ensemble-mean RMSE and Kendall's tau vs. ensemble
size), but the "seeds" being averaged are 10 MC-dropout forward passes of
the single seed-1 model (Fig2/results/mcseeds_results; base checkpoint =
seed-1 epoch010, the same model as seeds_results seed 1) instead of 10
independently-trained seeds. Ground truth: Fig2/results/test_panav.json
(leakage-filtered test set). Kendall's tau is computed the same way
as RMSE: between the M-sample ensemble-mean prediction and the ground
truth (scipy.stats.kendalltau(sub_mean, gt_vec)).

The y-axis ranges (RMSE and Kendall's tau) are fixed to match
seed_stability_10.png so the two figures are directly comparable.

Output: Fig2/Figs/mcdropout_stability_10.png
"""
import itertools
import json
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kendalltau


# ------------------------------------------------------------------ #
# Ground-truth / prediction loading -- identical to
# SI/mc_dropout/analyze_seed_stability.ipynb.
# ------------------------------------------------------------------ #
def parse_shift_list(lst, sequence, atom):
    vals, mask = [], []
    for i, x in enumerate(lst):
        residue = sequence[i]
        bad = False
        if x is None:
            bad = True
        elif isinstance(x, str):
            xs = x.strip().lower()
            if xs in {"none", "null", "nan", "inf", "+inf", "-inf"}:
                bad = True
            else:
                try:
                    x = float(x)
                except Exception:
                    bad = True
        if not bad and isinstance(x, (int, float)):
            if not math.isfinite(float(x)):
                bad = True
        if atom in ("N", "H") and residue == "P":
            bad = True
        if atom == "CB" and residue == "G":
            bad = True
        if bad:
            vals.append(0.0)
            mask.append(False)
        else:
            vals.append(float(x))
            mask.append(True)
    return vals, mask


def load_ground_truth(gt_path: Path, atom: str):
    with open(gt_path) as f:
        rbmrb = json.load(f)
    out = {}
    for ref_id, shift_lst in rbmrb[atom].items():
        if all(x is None or (isinstance(x, str) and x.lower() in ("none", "null"))
               for x in shift_lst):
            continue
        sequence = rbmrb["sequence"][ref_id]
        shift, mask = parse_shift_list(shift_lst, sequence, atom=atom)
        pdb_id = rbmrb["entry_ID"][ref_id].lower().removeprefix("bmr")
        out[pdb_id] = (shift, mask)
    return out


def load_pred(path):
    with open(path) as f:
        d = json.load(f)
    return d["pred"], d["mask"]


def build_matrix(pred_paths, gt_path, atom):
    per_seed = [load_pred(p) for p in pred_paths]  # [(pred_dict, mask_dict), ...]
    gt = load_ground_truth(Path(gt_path), atom=atom)

    common_pdbs = set(per_seed[0][0].keys())
    for pred_dict, _ in per_seed[1:]:
        common_pdbs &= set(pred_dict.keys())
    common_pdbs &= set(gt.keys())

    gt_vals, pred_cols = [], [[] for _ in per_seed]
    ids_used = 0
    for pdb_id in sorted(common_pdbs):
        true_shift, true_mask = gt[pdb_id]
        lens = [len(true_shift)]
        for pred_dict, mask_dict in per_seed:
            lens.append(len(pred_dict[pdb_id]))
            lens.append(len(mask_dict[pdb_id]))
        n = min(lens)
        if n == 0:
            continue

        combined_mask = np.array(true_mask[:n], dtype=bool)
        seed_masks = []
        for pred_dict, mask_dict in per_seed:
            seed_masks.append(np.array(mask_dict[pdb_id][:n], dtype=bool))
        for sm in seed_masks:
            combined_mask &= sm
        if not combined_mask.any():
            continue

        ids_used += 1
        gt_vals.append(np.array(true_shift[:n])[combined_mask])
        for k, (pred_dict, _) in enumerate(per_seed):
            pred_cols[k].append(np.array(pred_dict[pdb_id][:n])[combined_mask])

    all_pdbs = set(gt.keys())
    for pred_dict, _ in per_seed:
        all_pdbs |= set(pred_dict.keys())
    missing = len(all_pdbs) - ids_used
    gt_vec = np.concatenate(gt_vals)
    pred_mat = np.stack([np.concatenate(col) for col in pred_cols], axis=0)  # [n_samples, N]
    return pred_mat, gt_vec, ids_used, missing


def rmse_of(pred_vec, gt_vec):
    err = pred_vec - gt_vec
    return float(np.sqrt(np.mean(err ** 2)))


In [ ]:
# ------------------------------------------------------------------ #
# 10 MC-dropout forward passes of the seed-1 model, from
# Fig2/results/mcseeds_results (mcseed1-10).
# ------------------------------------------------------------------ #
import re
RESULTS_DIR = Path("../../../data/analysis/Fig2/results")
MC_DIR = RESULTS_DIR / "mcseeds_results"
_mc_of = lambda p: int(re.search(r"_mcseed(\d+)_", p.name).group(1))
pred_paths = sorted(MC_DIR.glob("predictions_N_panav_*_mcdropout_mcseed*_test.json"), key=_mc_of)
mc_seeds = [_mc_of(p) for p in pred_paths]
assert mc_seeds == list(range(1, 11)), f"expected mcseeds 1..10, found {mc_seeds}"
gt_path = RESULTS_DIR / "test_panav.json"
atom = "N"

OUT_DIR = Path("Figs")
OUT_DIR.mkdir(exist_ok=True)

assert len(pred_paths) == len(mc_seeds)

pred_mat, gt_vec, ids_used, missing = build_matrix(pred_paths, gt_path, atom)
n_seeds, n_res = pred_mat.shape
print(f"{ids_used} structures scored ({missing} skipped: missing from an MC-dropout "
      f"pass's output, gt, or fully masked), {n_res} (structure, residue) pairs total")

full_mean = pred_mat.mean(axis=0)
full_rmse = rmse_of(full_mean, gt_vec)
full_tau = kendalltau(full_mean, gt_vec).correlation
print(f"\nFull {n_seeds}-pass MC-dropout ensemble: RMSE={full_rmse:.4f}  Kendall's tau={full_tau:.4f}")


In [ ]:
# ---------------- plateau diagnostic (all subset sizes) ---------------- #
rows = []
for M in range(1, n_seeds + 1):
    rmses, taus = [], []
    for combo in itertools.combinations(range(n_seeds), M):
        sub_mean = pred_mat[list(combo)].mean(axis=0)
        rmses.append(rmse_of(sub_mean, gt_vec))
        tau = kendalltau(sub_mean, gt_vec).correlation
        taus.append(0.0 if np.isnan(tau) else tau)
    rows.append({
        "M": M,
        "rmse_mean": np.mean(rmses), "rmse_std": np.std(rmses),
        "kendall_tau_mean": np.mean(taus), "kendall_tau_std": np.std(taus),
    })
    print(f"M={M}  RMSE={np.mean(rmses):.4f}+/-{np.std(rmses):.4f}  "
          f"Kendall's tau={np.mean(taus):.4f}+/-{np.std(taus):.4f}")


In [ ]:
def set_style(base=12, family="Arial"):
    plt.rcParams.update({
        "font.family": family,
        "font.size": base,
        "axes.titlesize": base + 2,
        "axes.labelsize": base,
        "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2,
    })

set_style(base=14)

# ---------------- plot: single panel, RMSE + Kendall's tau on twin y-axes ---------------- #
# Fixed to the same ranges as seed_stability_10.png so the two figures are
# directly comparable at a glance.
RMSE_YLIM = (1.73, 1.85)
TAU_YLIM = (0.778, 0.796)

Ms = [r["M"] for r in rows]
rmse_mean = np.array([r["rmse_mean"] for r in rows]); rmse_std = np.array([r["rmse_std"] for r in rows])
tau_mean  = np.array([r["kendall_tau_mean"] for r in rows]); tau_std = np.array([r["kendall_tau_std"] for r in rows])

fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()

color_rmse = "#56B6C2"
color_tau = "#3F729B"

l1, = ax1.plot(Ms, rmse_mean, marker="o", color=color_rmse, lw=2, label="RMSE")
ax1.fill_between(Ms, rmse_mean - rmse_std, rmse_mean + rmse_std, color=color_rmse, alpha=0.3, lw=0)

l2, = ax2.plot(Ms, tau_mean, marker="s", color=color_tau, lw=2, label="Kendall's tau")
ax2.fill_between(Ms, tau_mean - tau_std, tau_mean + tau_std, color=color_tau, alpha=0.3, lw=0)

ax1.set_xlabel("number of MC-dropout passes averaged")
ax1.set_ylabel("ensemble-mean RMSE (ppm)", color=color_rmse)
ax1.tick_params(axis="y", labelcolor=color_rmse)
ax1.set_xticks(Ms)
ax1.set_ylim(*RMSE_YLIM)

ax2.set_ylabel("ensemble-mean Kendall's τ", color=color_tau)
ax2.tick_params(axis="y", labelcolor=color_tau)
ax2.set_ylim(*TAU_YLIM)

# ax1.set_title("RMSE and Kendall's tau vs. ensemble size")
ax1.legend(handles=[l1, l2], labels=["RMSE", "Kendall's tau"], loc="center right")

fig.tight_layout()
out_path = OUT_DIR / "mcdropout_stability_10.png"
fig.savefig(out_path, dpi=300)
fig.savefig(OUT_DIR / "mcdropout_stability_10.svg", dpi=300)
print(f"wrote {out_path}")
plt.show()